# GSM8K DAPO training on Colab

This notebook runs `RL_GRPO_train/configs/qwen25_3b_base_sft_dapo.yaml` from the LLM project. It uses the Qwen2.5-3B base SFT adapter, not the Instruct SFT adapter.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

candidates = [
    Path('/content/drive/MyDrive/LLM_project'),
    Path('/content/drive/MyDrive/LLM_project/project'),
    Path('/content/project'),
    Path.cwd(),
]
PROJECT_DIR = next((p for p in candidates if (p / 'RL_GRPO_train').exists() and (p / 'RL_common').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Cannot find LLM project root. Mount Drive or edit PROJECT_DIR manually.')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
!pwd

## Tokens and SwanLab

In [ ]:
import getpass
import os

USE_COLAB_SECRETS = True

if USE_COLAB_SECRETS:
    from google.colab import userdata
    for key in ['HF_TOKEN', 'SWANLAB_API_KEY']:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
else:
    for key in ['HF_TOKEN', 'SWANLAB_API_KEY']:
        if not os.environ.get(key):
            value = getpass.getpass(f'{key} (press Enter to skip): ')
            if value:
                os.environ[key] = value

os.environ.setdefault('SWANLAB_PROJECT', 'gsm8k-grpo')

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

print('SWANLAB_PROJECT =', os.environ.get('SWANLAB_PROJECT'))
print('HF_TOKEN set =', bool(os.environ.get('HF_TOKEN')))
print('SWANLAB_API_KEY set =', bool(os.environ.get('SWANLAB_API_KEY')))

## Install dependencies

In [ ]:
!pip uninstall -y torchao
!pip install -q -U "trl[vllm]>=0.29.0" "accelerate>=1.4.0" swanlab bitsandbytes sentencepiece
!pip install -q -e RL_common

import torch
import trl
import accelerate

try:
    import vllm
    print('vllm import ok')
except Exception as exc:
    raise RuntimeError('vLLM is required because the configs set grpo.use_vllm=true') from exc

print('torch =', torch.__version__)
print('trl =', trl.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))

## Select config

In [ ]:
import inspect
import yaml
from pathlib import Path
from trl import GRPOConfig

ALGO_NAME = 'dapo'
BASE_CONFIG = PROJECT_DIR / 'RL_GRPO_train/configs/qwen25_3b_base_sft_grpo.yaml'
ACTIVE_CONFIG = PROJECT_DIR / 'RL_GRPO_train/configs/qwen25_3b_base_sft_dapo.yaml'

ALLOWED_DIFFS = {('run', 'name'), ('grpo', 'run_name'), ('grpo', 'loss_type'), ('grpo', 'mask_truncated_completions'), ('grpo', 'epsilon'), ('grpo', 'epsilon_high'), ('reward', 'soft_overlong_punishment')}

def changed_paths(left, right, prefix=()):
    if isinstance(left, dict) and isinstance(right, dict):
        keys = set(left) | set(right)
        for key in sorted(keys):
            yield from changed_paths(left.get(key), right.get(key), prefix + (key,))
    elif left != right:
        yield prefix

base_cfg = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
cfg = yaml.safe_load(ACTIVE_CONFIG.read_text(encoding='utf-8'))

unexpected = sorted(set(changed_paths(base_cfg, cfg)) - ALLOWED_DIFFS)
if unexpected:
    raise AssertionError(f'Unexpected non-DAPO config changes: {unexpected}')

signature = inspect.signature(GRPOConfig.__init__)
accepts_kwargs = any(param.kind == inspect.Parameter.VAR_KEYWORD for param in signature.parameters.values())
required_args = {'loss_type', 'mask_truncated_completions', 'epsilon', 'epsilon_high'}
missing_args = set() if accepts_kwargs else required_args - set(signature.parameters)
if missing_args:
    raise RuntimeError(f'The installed TRL GRPOConfig does not support DAPO fields: {sorted(missing_args)}')

soft_cfg = cfg['reward']['soft_overlong_punishment']
if not soft_cfg.get('enabled', True):
    raise AssertionError('DAPO soft_overlong_punishment must be enabled')
if int(soft_cfg['max_completion_len']) != int(cfg['grpo']['max_completion_length']):
    raise AssertionError('soft_overlong max_completion_len must match grpo.max_completion_length')

print('ACTIVE_CONFIG =', ACTIVE_CONFIG)
print(ACTIVE_CONFIG.read_text(encoding='utf-8'))

## Train

In [ ]:
!accelerate launch --num_processes 1 RL_GRPO_train/train_grpo.py --config "{ACTIVE_CONFIG}"

## Inspect outputs

In [ ]:
import json
import sys

COMMON_SRC = PROJECT_DIR / 'RL_common/src'
if str(COMMON_SRC) not in sys.path:
    sys.path.insert(0, str(COMMON_SRC))

from rl_common.config import format_run_name, load_config, make_run_dir

cfg = load_config(str(ACTIVE_CONFIG))
cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)
run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
print('run_dir =', run_dir)
!find "{run_dir}" -maxdepth 2 -type f | sort

summary_path = run_dir / 'train_summary.json'
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8'))

best_path = run_dir / 'best_eval_results.json'
if best_path.exists():
    data = json.loads(best_path.read_text(encoding='utf-8'))
    print(json.dumps(data['summary'], indent=2, ensure_ascii=False))

## Optional final eval on official GSM8K test

Do not run this during training. This cell rewrites a temporary eval config so `adapter_path` points to the trained `final_adapter`; the tokenizer remains the base SFT tokenizer from the YAML.

In [ ]:
import sys
import yaml
from pathlib import Path

COMMON_SRC = PROJECT_DIR / 'RL_common/src'
if str(COMMON_SRC) not in sys.path:
    sys.path.insert(0, str(COMMON_SRC))

from rl_common.config import format_run_name, load_config, make_run_dir

RUN_FINAL_TEST = True

if RUN_FINAL_TEST:
    cfg = load_config(str(ACTIVE_CONFIG))
    cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)
    train_run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
    final_adapter = train_run_dir / 'final_adapter'
    if not final_adapter.exists():
        raise FileNotFoundError(f'Missing final adapter: {final_adapter}')

    cfg['model']['adapter_path'] = str(final_adapter)
    cfg['run']['name'] = cfg['run']['resolved_name'] + '_final_test'
    cfg['eval']['max_new_tokens'] = 256
    cfg['eval']['batch_size'] = 512

    eval_cfg = Path(f'/content/{ALGO_NAME}_final_eval.yaml')
    eval_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print(eval_cfg.read_text(encoding='utf-8')[:1500])
    !python RL_GRPO_train/train_grpo.py --config "{eval_cfg}" --eval-only --final-test